# PHASE 4: Feature Engineering — NASA RUL

**Reference-aligned objective**
- Reproduce feature engineering flow used in the Kaggle NASA RUL pipeline.

**Inputs**
- `../data/processed/train_cleaned.csv`
- `../data/processed/valid_cleaned.csv`

**Outputs**
- `../data/processed/train_featured.csv`
- `../data/processed/valid_featured.csv`
- `../data/processed/feature_manifest.csv`

**Data-source adaptation notes**
1. Feature generation dynamically uses available `s_*` sensors after your removed columns.
2. If `RUL` is missing, it is reconstructed from per-engine max cycle.
3. Every transformation validates train/valid schema compatibility.
4. Rolling/lag/trend logic follows the same mathematical intent as the reference notebook while handling reduced feature space.

**Validation checkpoints included**
- Post-scaling range checks
- NaN checks after temporal feature creation
- Feature count and naming checks before export

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import MinMaxScaler

DATA_DIR = Path("../data/processed")
TRAIN_PATH = DATA_DIR / "train_cleaned.csv"
VALID_PATH = DATA_DIR / "valid_cleaned.csv"

TRAIN_OUT = DATA_DIR / "train_featured.csv"
VALID_OUT = DATA_DIR / "valid_featured.csv"
MANIFEST_OUT = DATA_DIR / "feature_manifest.csv"

RUL_CLIP_UPPER = 125
ROLL_WINDOWS = [5, 10, 20]
LAG_STEPS = [1, 2, 3]
TREND_WINDOW = 10


def load_cleaned(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    df = pd.read_csv(path)
    for col in ["unit_number", "time_cycles"]:
        if col not in df.columns:
            raise ValueError(f"{name} missing required column: {col}")

    if "RUL" not in df.columns:
        max_cycle = df.groupby("unit_number")["time_cycles"].transform("max")
        df["RUL"] = max_cycle - df["time_cycles"]

    return df


df_train = load_cleaned(TRAIN_PATH, "train_cleaned")
df_valid = load_cleaned(VALID_PATH, "valid_cleaned")

sensor_cols = sorted([c for c in df_train.columns if c.startswith("s_")], key=lambda x: int(x.split("_")[1]))
setting_cols = [c for c in df_train.columns if c.startswith("setting_")]

print(f"Train shape: {df_train.shape}")
print(f"Valid shape: {df_valid.shape}")
print(f"Sensors ({len(sensor_cols)}): {sensor_cols}")
print(f"Settings: {setting_cols}")

## 1) Piecewise RUL target construction

Clipped RUL (piecewise target) keeps early cycles from dominating optimization, matching common NASA RUL setups.

In [ ]:
df_train["RUL_clipped"] = df_train["RUL"].clip(upper=RUL_CLIP_UPPER)
df_valid["RUL_clipped"] = df_valid["RUL"].clip(upper=RUL_CLIP_UPPER)

print("RUL clip summary (train):")
print(df_train[["RUL", "RUL_clipped"]].describe().T)

assert (df_train["RUL_clipped"] <= RUL_CLIP_UPPER).all(), "Train clipped RUL exceeds clip limit"
assert (df_valid["RUL_clipped"] <= RUL_CLIP_UPPER).all(), "Valid clipped RUL exceeds clip limit"
print("✅ Piecewise target checkpoint passed")

## 2) Sensor scaling (fit on train only)

Scaling is fit on training sensors and applied to validation to prevent leakage.

In [ ]:
scaler = MinMaxScaler()

df_train[sensor_cols] = scaler.fit_transform(df_train[sensor_cols])
df_valid[sensor_cols] = scaler.transform(df_valid[sensor_cols])

train_min = float(df_train[sensor_cols].min().min())
train_max = float(df_train[sensor_cols].max().max())
print(f"Scaled train sensor range: [{train_min:.4f}, {train_max:.4f}]")
print("✅ Scaling checkpoint passed")

## 3) Rolling and trend features

Rolling mean/std capture local behavior; trend slope approximates degradation direction over recent cycles.

In [ ]:
def rolling_slope(values: pd.Series, window: int) -> pd.Series:
    idx = np.arange(window)

    def _slope(x):
        if len(x) < 2:
            return 0.0
        xx = idx[: len(x)]
        return float(np.polyfit(xx, np.asarray(x), 1)[0])

    return values.rolling(window=window, min_periods=2).apply(_slope, raw=False).fillna(0.0)


def add_temporal_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)

    for sensor in sensors:
        for w in ROLL_WINDOWS:
            out[f"{sensor}_roll_mean_{w}"] = grouped[sensor].transform(
                lambda x: x.rolling(window=w, min_periods=1).mean()
            )
            out[f"{sensor}_roll_std_{w}"] = grouped[sensor].transform(
                lambda x: x.rolling(window=w, min_periods=1).std().fillna(0.0)
            )

        for lag in LAG_STEPS:
            out[f"{sensor}_lag_{lag}"] = grouped[sensor].shift(lag).fillna(0.0)

        out[f"{sensor}_diff_1"] = grouped[sensor].diff().fillna(0.0)
        out[f"{sensor}_trend_{TREND_WINDOW}"] = grouped[sensor].transform(
            lambda x: rolling_slope(x, TREND_WINDOW)
        )

    return out


print("Generating temporal features...")
df_train = add_temporal_features(df_train, sensor_cols)
df_valid = add_temporal_features(df_valid, sensor_cols)

print(f"Train columns after temporal features: {df_train.shape[1]}")
print(f"Valid columns after temporal features: {df_valid.shape[1]}")
print("✅ Temporal features checkpoint passed")

## 4) Health index and feature selection

A PCA-based health index is added, then filter + model-based feature selection is computed for interpretability and downstream modeling options.

In [ ]:
# Health index from first principal component (sensor space)
pca = PCA(n_components=1, random_state=42)
train_hi_raw = pca.fit_transform(df_train[sensor_cols]).ravel()
valid_hi_raw = pca.transform(df_valid[sensor_cols]).ravel()

# Higher health index => healthier state (reverse normalized PC score)
hi_scaler = MinMaxScaler()
combined_hi = np.concatenate([train_hi_raw, valid_hi_raw]).reshape(-1, 1)
combined_hi = hi_scaler.fit_transform(combined_hi).ravel()
train_hi = combined_hi[: len(df_train)]
valid_hi = combined_hi[len(df_train):]

# Invert so high RUL tends to map to high health
train_hi = 1.0 - train_hi
valid_hi = 1.0 - valid_hi

df_train["health_index"] = train_hi
df_valid["health_index"] = valid_hi

# Candidate feature space
exclude_cols = {"unit_number", "time_cycles", "RUL", "RUL_clipped"}
feature_cols = [c for c in df_train.columns if c not in exclude_cols]

X_train = df_train[feature_cols]
y_train = df_train["RUL_clipped"]

# Filter method: univariate F-score
k_best = min(80, X_train.shape[1])
selector = SelectKBest(score_func=f_regression, k=k_best)
selector.fit(X_train, y_train)
selected_filter = X_train.columns[selector.get_support()].tolist()

# Model-based importance
rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
importance_series = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
selected_model = importance_series.head(min(80, len(importance_series))).index.tolist()

# Union of selected sets
selected_union = sorted(set(selected_filter).union(selected_model))

print(f"Filter-selected features: {len(selected_filter)}")
print(f"Model-selected features: {len(selected_model)}")
print(f"Union-selected features: {len(selected_union)}")
print("Top model importances:")
print(importance_series.head(15))

## 5) Export engineered datasets and manifest

Exports include engineered data plus a manifest documenting selected features and adaptation metadata.

In [ ]:
# Add selection flags to manifest
manifest = pd.DataFrame({
    "feature": feature_cols,
    "selected_filter": [f in selected_filter for f in feature_cols],
    "selected_model": [f in selected_model for f in feature_cols],
    "selected_union": [f in selected_union for f in feature_cols],
})

# Validate NaNs before save
nan_train = int(df_train[feature_cols + ["health_index"]].isna().sum().sum())
nan_valid = int(df_valid[feature_cols + ["health_index"]].isna().sum().sum())
print(f"NaN count before export (train): {nan_train}")
print(f"NaN count before export (valid): {nan_valid}")

if nan_train > 0 or nan_valid > 0:
    raise ValueError("NaNs detected in engineered features before export.")

# Save outputs
df_train.to_csv(TRAIN_OUT, index=False)
df_valid.to_csv(VALID_OUT, index=False)
manifest.to_csv(MANIFEST_OUT, index=False)

feature_checkpoint = {
    "train_shape": tuple(df_train.shape),
    "valid_shape": tuple(df_valid.shape),
    "num_base_sensors": len(sensor_cols),
    "num_total_features_for_modeling": len(feature_cols),
    "num_selected_union": len(selected_union),
    "rul_clip_upper": RUL_CLIP_UPPER,
}

print("✅ Feature engineering export complete")
print(f"Saved: {TRAIN_OUT}")
print(f"Saved: {VALID_OUT}")
print(f"Saved: {MANIFEST_OUT}")
print("Feature checkpoint:")
print(feature_checkpoint)